# CSCE 633 | Improving CLIP on FASTER

This notebook uses the course's [starter repository](https://github.com/xywei00/csce689_iSogCLR/tree/project) and the four [course archives](https://drive.google.com/drive/folders/1Wh-X0YHUoJCRdIS2hVFtRC6xU5JToqvz). CC3M is the only training split. MSCOCO and ImageNet are used for validation. The graded runs use pretrained timm ResNet-50 and Hugging Face DistilBERT, batch size 128, and 30 epochs.

### 1. Set up FASTER and open Jupyter

In [FASTER OnDemand](https://portal-faster.hprc.tamu.edu), use **Files → Scratch Directory**. Create a folder called `csce633_clip` and an `archives` subfolder. Put the course archives in `archives` and upload this notebook into `csce633_clip`. If you uploaded the archives elsewhere, edit `ARCHIVES` in the next cell. Run `showquota` in a FASTER shell to check storage and file counts before extraction. The training images alone can use about 100,000 files.

Open a FASTER shell and run the following once. Select an available Anaconda module from the output of the first command. This creates a separate training environment, so the Jupyter app itself can use its default Python kernel.

```bash
module spider Anaconda3
module load Anaconda3/<version-listed-on-FASTER>
mkdir -p "$SCRATCH/csce633_clip"
conda create -y -p "$SCRATCH/csce633_clip/clip_env" python=3.10 pip
CLIP_PY="$SCRATCH/csce633_clip/clip_env/bin/python"
"$CLIP_PY" -m pip install torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
"$CLIP_PY" -m pip install "numpy==1.26.4" "Pillow==10.0.1" "opencv-python-headless==4.10.0.84" "timm==0.9.7" "transformers==4.34.0" "huggingface-hub==0.16.4" "tqdm==4.66.1"
```

In OnDemand select **Interactive Apps → JupyterLab** or **Jupyter Notebook**, launch a CPU session, and open this file from Scratch. The notebook submits separate GPU jobs. You can also submit the generated scripts from a FASTER shell if your Jupyter compute node cannot run `sbatch`. The first smoke job needs access to the pretrained weights. If the GPU job cannot download them, use the FASTER shell to pre-cache them with the command printed in Step 4.

### 2. Locate the archives and clone the course starter

`clip_train.tar.gz` contains annotations. `cc3m_subset_100k.tar.gz` contains the matching training images. `mscoco_val.tar.gz` and `val.tar` contain validation images. The Drive listing shows roughly 9.9 GB of compressed files in total.

In [ ]:
import json
import os
import re
import shlex
import subprocess
from pathlib import Path

ROOT = (Path(os.environ["SCRATCH"]) / "csce633_clip").resolve()
ARCHIVES = ROOT / "archives"  # Edit this if you uploaded to another folder.
DATA = ROOT / "datasets"
ANN = ROOT / "clip_train"
REPO = ROOT / "iSogCLR"
TRAIN_PYTHON = ROOT / "clip_env/bin/python"
SCRIPT = REPO / "bimodal_exps/clip.py"
CACHE = ROOT / "model_cache"
JOBS = ROOT / "jobs"
OUTPUT = ROOT / "output"
STARTER_COMMIT = "4466249720ea6be09cd42b2f8aec1111fb504145"

for directory in (ROOT, DATA, CACHE, JOBS, OUTPUT):
    directory.mkdir(parents=True, exist_ok=True)

archive_targets = {
    "clip_train.tar.gz": ROOT,
    "cc3m_subset_100k.tar.gz": DATA,
    "mscoco_val.tar.gz": DATA,
    "val.tar": DATA / "imagenet",
}

for name in archive_targets:
    archive = ARCHIVES / name
    if not archive.is_file():
        raise FileNotFoundError(f"Missing {archive}. Check ARCHIVES and the uploaded filename.")
    print(f"{name:27s} {archive.stat().st_size / 1e6:,.1f} MB")

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "-b", "project",
         "https://github.com/xywei00/csce689_iSogCLR.git", str(REPO)],
        check=True,
    )
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", STARTER_COMMIT], check=True)
print("Starter commit:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip())

### 3. Extract once and verify the paths

Run this on the four original course archives. Completion markers let you rerun this cell after an interrupted extraction. For ImageNet, the starter expects an ImageFolder tree with class folders. If `val.tar` contains flat images or the class check fails, inspect its layout before evaluating. ImageNet class IDs must match the starter's class order for top-1 accuracy to be meaningful.

In [ ]:
for filename, destination in archive_targets.items():
    destination.mkdir(parents=True, exist_ok=True)
    marker = ROOT / f".extracted-{filename}"
    archive = ARCHIVES / filename
    stamp = f"{archive.resolve()} {archive.stat().st_size} {archive.stat().st_mtime_ns}"
    if marker.exists() and marker.read_text() == stamp:
        print("Already extracted:", filename)
        continue
    print("Extracting:", filename, flush=True)
    tar_flag = "-xzf" if filename.endswith(".gz") else "-xf"
    subprocess.run(["tar", tar_flag, str(archive), "-C", str(destination)], check=True)
    marker.write_text(stamp)

for annotation_file, image_root in [
    (ANN / "cc3m_train_subset.json", DATA / "cc3m_subset_100k"),
    (ANN / "coco_val.json", DATA / "mscoco_val/mscoco_val2014_subset_5k"),
]:
    with annotation_file.open() as handle:
        records = json.load(handle)
    print(annotation_file.name, len(records), "records")
    for row in records[:5] + records[-5:]:
        image = image_root / row["image"]
        if not image.is_file():
            raise FileNotFoundError(f"Annotation refers to missing image: {image}")
    print("Sample image:", image)

imagenet_base = DATA / "imagenet"
IMAGENET_VAL = imagenet_base / "val"
if not IMAGENET_VAL.is_dir() and any(
    p.is_dir() and re.fullmatch(r"n[0-9]{8}", p.name) for p in imagenet_base.iterdir()
):
    IMAGENET_VAL = imagenet_base
class_folders = sorted(p.name for p in IMAGENET_VAL.iterdir() if p.is_dir()) if IMAGENET_VAL.is_dir() else []
assert len(class_folders) == 1000 and all(re.fullmatch(r"n[0-9]{8}", name) for name in class_folders), (
    f"Expected 1,000 ImageNet synset folders under {IMAGENET_VAL}. "
    "Check the extracted val.tar structure and verify class ordering before evaluation."
)
print("ImageNet validation:", IMAGENET_VAL, "| classes:", len(class_folders))

### 4. Define the experiments and write GPU job scripts

The four runs isolate each comparison. Compare three losses using AdamW, then compare AdamW and Adam using SogCLR. All runs use the same seed and starting optimizer learning rate. Keep the two optimizer names distinct from the loss names. The starter uses `--temp` for fixed CLIP/SogCLR temperature and `--tau_init` for the initial iSogCLR per-example temperature.

The ImageNet evaluator uses a fixed batch size of 512 in the starter, so it can have a larger GPU memory requirement than training. The defaults below request an A100 and 64 GB of CPU RAM. Set `ACCOUNT` only if Slurm needs your assigned allocation specified. Set `FULL_TIME` using actual timing from the smoke run and the first full epoch.

In [ ]:
assert TRAIN_PYTHON.is_file(), "Create clip_env using Step 1 on FASTER."
assert SCRIPT.is_file(), "Clone the starter in Step 2."

EXPERIMENTS = {
    "clip_adamw": ("clip", "adamw"),
    "sogclr_adamw": ("sogclr", "adamw"),
    "isogclr_adamw": ("isogclr_new", "adamw"),
    "sogclr_adam": ("sogclr", "adam"),
}
ACCOUNT = None  # Example: "your_course_allocation"
GPU = "a100"
FULL_TIME = "36:00:00"
BASE = [
    str(TRAIN_PYTHON), "-u", str(SCRIPT),
    "--data_path", str(DATA), "--ann_path", str(ANN),
    "--train_file", "cc3m_train_subset.json", "--train_image_root", "cc3m_subset_100k",
    "--image_encoder", "resnet50", "--text_encoder", "distilbert-base-uncased",
    "--init_model", "--use_amp", "--no-distributed",
    "--batch_size_train", "128", "--batch_size_test", "128",
    "--seed", "42", "--lr", "0.0002", "--sched", "cosine",
    "--temp", "0.01", "--tau_init", "0.01", "--sogclr_gamma", "0.8",
]

def experiment_command(name):
    loss, optimizer = EXPERIMENTS[name]
    return BASE + ["--ita_type", loss, "--opt", optimizer]

def write_job(name, commands, time_limit):
    script = JOBS / f"{name}.sh"
    directives = [
        "#!/bin/bash",
        f"#SBATCH --job-name={name}",
        "#SBATCH --partition=gpu",
        f"#SBATCH --gres=gpu:{GPU}:1",
        "#SBATCH --nodes=1",
        "#SBATCH --ntasks=1",
        "#SBATCH --cpus-per-task=12",
        "#SBATCH --mem=64G",
        f"#SBATCH --time={time_limit}",
        f'#SBATCH --output="{JOBS}/%x-%j.log"',
    ]
    if ACCOUNT:
        directives.append(f"#SBATCH --account={ACCOUNT}")
    preamble = [
        "set -euo pipefail",
        f"cd {shlex.quote(str(REPO))}",
        f"export HF_HOME={shlex.quote(str(CACHE / 'huggingface'))}",
        f"export TORCH_HOME={shlex.quote(str(CACHE / 'torch'))}",
        "export OMP_NUM_THREADS=1",
        "nvidia-smi",
    ]
    script.write_text("\n".join(directives + preamble + [shlex.join(cmd) for cmd in commands]) + "\n")
    return script

def submit(script):
    response = subprocess.check_output(["sbatch", "--parsable", str(script)], text=True).strip()
    job_id = response.split(";")[0]
    print(f"Submitted job {job_id}. Logs are under {JOBS}")
    return job_id

def eval_args(checkpoint, destination):
    return [
        "--evaluate", "--checkpoint", str(checkpoint),
        "--zs_dataset", "imagenet", "--zs_datafolder", str(IMAGENET_VAL),
        "--output_dir", str(destination),
    ]

# Three full minibatches for 100k records. This result is only a pipeline check.
SMOKE_DIR = OUTPUT / "smoke_sogclr"
smoke_base = experiment_command("sogclr_adamw") + ["--epochs", "1", "--train_frac", "0.005"]
smoke_script = write_job("clip_smoke", [
    smoke_base + ["--output_dir", str(SMOKE_DIR)],
    smoke_base + eval_args(SMOKE_DIR / "checkpoint_1.pth", SMOKE_DIR / "validation"),
], "02:00:00")
print("Smoke script:", smoke_script)
print("If the GPU cannot download weights, pre-cache on the FASTER shell:")
cache_download = (
    'import timm; from transformers import AutoModel, AutoTokenizer; '
    'timm.create_model("resnet50", pretrained=True); '
    'AutoTokenizer.from_pretrained("distilbert-base-uncased"); '
    'AutoModel.from_pretrained("distilbert-base-uncased")'
)
print(
    f"HF_HOME={shlex.quote(str(CACHE / 'huggingface'))} "
    f"TORCH_HOME={shlex.quote(str(CACHE / 'torch'))} "
    f"{shlex.join([str(TRAIN_PYTHON), '-c', cache_download])}"
)

### 5. Submit the smoke test and inspect it before training

Change `SUBMIT_SMOKE` to `True` once. Save the printed job ID. Check OnDemand **Jobs → Active Jobs** or run `squeue -u "$USER"` in a FASTER shell. To inspect the job, open its log under `jobs/` or use `tail -f` on it. Wait for both `coco val:` and `zeroshot:` and check that the job completed. The smoke score is expected to be low.

In [ ]:
SUBMIT_SMOKE = False
if SUBMIT_SMOKE:
    smoke_job = submit(smoke_script)
else:
    print("Enable SUBMIT_SMOKE once, or run:", shlex.join(["sbatch", str(smoke_script)]))

### 6. Run the four 30-epoch comparisons

Start with SogCLR + AdamW. Confirm its first epoch works, then submit the others as your allocation permits. For each run, the job trains first and evaluates `checkpoint_30.pth` on both validation sets. The notebook submission cell directs you to the evaluation-only script when that final checkpoint already exists. The starter's per-epoch files hold model weights only, so an interrupted training run needs a fresh run for an equivalent 30-epoch experiment. Each model checkpoint is roughly 365 MB, so budget about 11 GB per experiment for its 30 checkpoints plus logs. Time-limit errors are visible via `sacct -j JOB_ID --format=JobID,State,Elapsed,ExitCode`.

In [ ]:
scripts = {}
for name in EXPERIMENTS:
    run_dir = OUTPUT / name
    checkpoint = run_dir / "checkpoint_30.pth"
    base = experiment_command(name) + ["--epochs", "30"]
    train_cmd = base + ["--output_dir", str(run_dir)]
    eval_cmd = base + eval_args(checkpoint, run_dir / "validation")
    # Keep training and validation in the same GPU allocation.
    # A separate evaluation script also lets you retry validation independently.
    scripts[name] = write_job(name, [train_cmd, eval_cmd], FULL_TIME)
    write_job(f"{name}_eval", [eval_cmd], "02:00:00")

RUN_TO_SUBMIT = None  # Change to one key above, e.g. "sogclr_adamw", and run once.
for name, script in scripts.items():
    print(name, shlex.join(["sbatch", str(script)]))
if RUN_TO_SUBMIT:
    run_dir = OUTPUT / RUN_TO_SUBMIT
    if (run_dir / "checkpoint_30.pth").is_file():
        print("Training checkpoint exists. Submit the evaluation-only script if needed:")
        print(shlex.join(["sbatch", str(JOBS / f"{RUN_TO_SUBMIT}_eval.sh")]))
    else:
        submit(scripts[RUN_TO_SUBMIT])

### 7. Calculate the project score and select a checkpoint

The starter logs `val_txt_r1` for image-to-text and `val_img_r1` for text-to-image. Its ImageNet top-1 key is `zeroshot_top1`. All three are percentages. The course score is their unweighted mean. Select the best completed full-data run using this score after all four comparisons have finished. In the report, include the three metrics, their mean, the loss and optimizer for each row, your method, and your team's contributions. The course also asks for training code, trained model weights, a report of at least five pages, and a 10-minute presentation video.

In [ ]:
import csv
import math

def last_record(path, required_keys):
    if not path.is_file():
        return None
    for line in reversed(path.read_text().splitlines()):
        if line.startswith("{"):
            record = json.loads(line)
            if required_keys <= record.keys():
                return record
    return None

results = []
for name, (loss, optimizer) in EXPERIMENTS.items():
    directory = OUTPUT / name
    val = directory / "validation"
    coco = last_record(val / "coco_log.txt", {"val_txt_r1", "val_img_r1"})
    imagenet = last_record(val / "zeroshot_imagenet_log.txt", {"zeroshot_top1"})
    checkpoint = directory / "checkpoint_30.pth"
    if coco is None or imagenet is None or not checkpoint.is_file():
        print("Pending:", name)
        continue
    scores = (coco["val_txt_r1"], coco["val_img_r1"], imagenet["zeroshot_top1"])
    if not all(math.isfinite(score) and 0 <= score <= 100 for score in scores):
        raise ValueError(f"Invalid validation metrics for {name}: {scores}")
    results.append(dict(
        run=name, loss=loss, optimizer=optimizer, image_to_text_r1=scores[0],
        text_to_image_r1=scores[1], imagenet_top1=scores[2],
        average=sum(scores) / 3, checkpoint=str(checkpoint),
    ))

results.sort(key=lambda result: result["average"], reverse=True)
if results:
    table_path = ROOT / "validation_results.csv"
    with table_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(results[0]))
        writer.writeheader()
        writer.writerows(results)
    for row in results:
        print(f"{row['run']:20s} {row['image_to_text_r1']:6.2f} "
              f"{row['text_to_image_r1']:6.2f} {row['imagenet_top1']:6.2f} "
              f"mean={row['average']:.2f}")
    print("Results CSV:", table_path)
if len(results) == len(EXPERIMENTS):
    best = results[0]
    print("Selected model:", best["checkpoint"])
    print("Training config:", OUTPUT / best["run"] / "args.json")

### 8. Prepare the submission

Retain the chosen `checkpoint_30.pth` and its `args.json`, the matching generated training script and notebook, the course code at the printed commit plus any edits, and `validation_results.csv`. The checkpoint stores the model state under the `model` key. The matching evaluation script shows how to reload it. Copy the selected model and results out of FASTER Scratch before the project ends. Include the validation comparison table in your report and explain which comparisons hold the optimizer or the loss fixed.